# Particle Flow Field

This sketch creates a mesmerizing visualization of particles flowing through a dynamic Perlin noise-based flow field. The particles leave colorful trails as they move, creating organic, fluid patterns.

**Features:**
- 800 particles following a flow field generated with Perlin noise
- Color-changing trails based on particle speed
- Mouse interaction: the flow field responds to mouse position
- Click to create burst effects with new particles

In [ ]:
// Global variables for particles and flow field
const particles = [];
const flowField = [];
const cellSize = 20;
let cols, rows;
let zoff = 0;
let hueBase = 0;

## Particle Class

Each particle has its own position, velocity, and acceleration. Particles follow the flow field vectors and create colorful trails. They can also be created as "burst" particles with limited lifetime when clicking.

In [ ]:
class Particle {
    constructor() {
        this.pos = createVector(random(width), random(height));
        this.vel = createVector(0, 0);
        this.acc = createVector(0, 0);
        this.maxSpeed = random(2, 4);
        this.prevPos = this.pos.copy();
        this.hueOffset = random(60);
        this.size = random(1, 3);
        this.isBurst = false;
        this.life = 255;
    }

    follow(vectors) {
        const x = floor(this.pos.x / cellSize);
        const y = floor(this.pos.y / cellSize);
        const index = x + y * cols;
        const force = vectors[index];
        if (force) {
            this.applyForce(force);
        }
    }

    applyForce(force) {
        this.acc.add(force);
    }

    update() {
        this.vel.add(this.acc);
        this.vel.limit(this.maxSpeed);
        this.pos.add(this.vel);
        this.acc.mult(0);

        if (this.isBurst) {
            this.life -= 3;
            this.vel.mult(0.98);
        }
    }

    edges() {
        if (this.pos.x > width) {
            this.pos.x = 0;
            this.prevPos.x = 0;
        }
        if (this.pos.x < 0) {
            this.pos.x = width;
            this.prevPos.x = width;
        }
        if (this.pos.y > height) {
            this.pos.y = 0;
            this.prevPos.y = 0;
        }
        if (this.pos.y < 0) {
            this.pos.y = height;
            this.prevPos.y = height;
        }
    }

    show() {
        const hue = (hueBase + this.hueOffset) % 360;
        const speed = this.vel.mag();
        const saturation = map(speed, 0, this.maxSpeed, 40, 90);
        const brightness = map(speed, 0, this.maxSpeed, 60, 100);
        const alpha = this.isBurst ? this.life : 80;

        stroke(hue, saturation, brightness, alpha);
        strokeWeight(this.size);

        const d = dist(this.pos.x, this.pos.y, this.prevPos.x, this.prevPos.y);
        if (d < 50) {
            line(this.pos.x, this.pos.y, this.prevPos.x, this.prevPos.y);
        }

        this.prevPos = this.pos.copy();
    }
}

## Setup

Initialize the canvas with HSB color mode for easy color manipulation. Create the flow field grid and populate it with 800 particles.

In [ ]:
function setup() {
    const canvas = createCanvas(innerWidth, innerHeight);
    colorMode(HSB, 360, 100, 100, 100);

    cols = floor(width / cellSize) || 45;
    rows = floor(height / cellSize) || 30;

    for (let i = 0; i < cols * rows; i++) {
        flowField.push(null);
    }

    for (let i = 0; i < 800; i++) {
        particles.push(new Particle());
    }

    background(240, 30, 8);
}

## Animation Loop

The draw function continuously updates the flow field using Perlin noise and animates all particles. The flow field responds to mouse position, creating interactive distortions within a 150-pixel radius.

In [ ]:
function draw() {
    background(240, 30, 8, 3);

    let yoff = 0;
    for (let y = 0; y < rows; y++) {
        let xoff = 0;
        for (let x = 0; x < cols; x++) {
            const index = x + y * cols;
            let angle = noise(xoff, yoff, zoff) * TWO_PI * 2;

            const d = dist(x * cellSize, y * cellSize, mouseX, mouseY);
            if (d < 150) {
                const mouseAngle = atan2(y * cellSize - mouseY, x * cellSize - mouseX);
                angle = lerp(angle, mouseAngle + PI, map(d, 0, 150, 0.8, 0));
            }

            flowField[index] = p5.Vector.fromAngle(angle);
            flowField[index].setMag(1);
            xoff += 0.1;
        }
        yoff += 0.1;
    }
    zoff += 0.003;
    hueBase += 0.3;

    for (const particle of particles) {
        particle.follow(flowField);
        particle.update();
        particle.edges();
        particle.show();
    }
}

## Mouse Interaction

Click anywhere to create a burst of 50 new particles that radiate outward from the click point. These burst particles have limited lifetime and gradually fade out.

In [ ]:
function mousePressed() {
    for (let i = 0; i < 50; i++) {
        const newParticle = new Particle();
        newParticle.pos = createVector(mouseX, mouseY);
        newParticle.vel = p5.Vector.random2D().mult(random(2, 8));
        newParticle.isBurst = true;
        particles.push(newParticle);
    }
    if (particles.length > 1200) {
        particles.splice(0, 50);
    }
}

In [ ]:
%show